# Semantic Kernel 

ഈ കോഡ് സാമ്പിളിൽ, നിങ്ങൾ ഒരു ലളിതമായ ഏജന്റ് സൃഷ്‌ടിക്കാൻ [Semantic Kernel](https://aka.ms/ai-agents-beginners/semantic-kernel) AI ഫ്രെയിംവർക്ക് ഉപയോഗിക്കും.

വിവിധ ഏജന്റിക് മാതൃകകൾ നടപ്പിലാക്കുമ്പോൾ പിന്നീട് ആപ്ലൈ ചെയ്യുന്ന ഘട്ടങ്ങൾ കാണിക്കുന്നതിനാണ് ഈ സാമ്പിൾ ഉദ്ദേശിക്കുന്നത്.


## ആവശ്യമായ Python പാക്കേജുകൾ ഇറക്കുമതി ചെയ്യുക


In [ ]:
import json
import os 

from typing import Annotated

from dotenv import load_dotenv

from IPython.display import display, HTML

from openai import AsyncOpenAI

from semantic_kernel.agents import ChatCompletionAgent, ChatHistoryAgentThread
from semantic_kernel.connectors.ai.open_ai import OpenAIChatCompletion
from semantic_kernel.contents import FunctionCallContent, FunctionResultContent, StreamingTextContent
from semantic_kernel.functions import kernel_function

## ക്ലയന്റ് സൃഷ്ടിക്കൽ

ഈ ഉദാഹരണത്തിൽ, LLM ആക്‌സസ് ചെയ്യാൻ [GitHub മോഡലുകൾ](https://aka.ms/ai-agents-beginners/github-models) ഉപയോഗിക്കും.

`ai_model_id` `gpt-4o-mini` ആയി സജ്ജമാക്കി. വ്യത്യസ്ത ഫലങ്ങൾ കാണാൻ GitHub മോഡലുകളുടെ മാർക്കറ്റ്‌പ്ലേസ് ਵਿੱਚ ലഭ്യമായ മറ്റൊരു മോഡലിലേക്ക് മാറ്റി നോക്കുക.

GitHub മോഡലുകളുടെ `base_url` కోసం ആവശ്യമായ `Azure Inference SDK` ഉപയോഗിക്കാൻ, Semantic Kernel എന്നതിലുളള `OpenAIChatCompletion` കണക്റ്റർ ഉപയോഗിക്കാം. Semantic Kernel മറ്റ് മോഡൽ പ്രൊവൈഡർമാരോടുകൂടെ ഉപയോഗിക്കാൻ സാധിക്കുന്ന മറ്റു [ലഭ്യമായ കണക്റ്ററുകളും](https://learn.microsoft.com/semantic-kernel/concepts/ai-services/chat-completion) ഉണ്ട്.


In [ ]:
import random   

# Define a sample plugin for the sample

class DestinationsPlugin:
    """A List of Random Destinations for a vacation."""

    def __init__(self):
        # List of vacation destinations
        self.destinations = [
            "Barcelona, Spain",
            "Paris, France",
            "Berlin, Germany",
            "Tokyo, Japan",
            "Sydney, Australia",
            "New York, USA",
            "Cairo, Egypt",
            "Cape Town, South Africa",
            "Rio de Janeiro, Brazil",
            "Bali, Indonesia"
        ]
        # Track last destination to avoid repeats
        self.last_destination = None

    @kernel_function(description="Provides a random vacation destination.")
    def get_random_destination(self) -> Annotated[str, "Returns a random vacation destination."]:
        # Get available destinations (excluding last one if possible)
        available_destinations = self.destinations.copy()
        if self.last_destination and len(available_destinations) > 1:
            available_destinations.remove(self.last_destination)

        # Select a random destination
        destination = random.choice(available_destinations)

        # Update the last destination
        self.last_destination = destination

        return destination

In [ ]:
load_dotenv()
client = AsyncOpenAI(
    api_key=os.environ.get("GITHUB_TOKEN"), 
    base_url="https://models.inference.ai.azure.com/",
)

# Create an AI Service that will be used by the `ChatCompletionAgent`
chat_completion_service = OpenAIChatCompletion(
    ai_model_id="gpt-4o-mini",
    async_client=client,
)

## ഏജന്റ് സൃഷ്ടിക്കല്‍

ഈ സ്ഥിതിയില്‍,  `TravelAgent` എന്ന പേരിലുള്ള ഏജന്റ് സൃഷ്ടിക്കുന്നു.

ഈ ഉദാഹരണത്തില്‍, നാം വളരെ അടിസ്ഥാന ഉപദേശം ഉപയോഗിക്കുന്നു. ഏജന്റിന്റെ പെരുമാറ്റം എങ്ങനെ മാറുന്നുവെന്ന് കാണാന്‍ ഈ ഉപദേശം സ്വതന്ത്രമായി മാറ്റാവുന്നതാണ്.


In [ ]:
agent = ChatCompletionAgent(
    service=chat_completion_service, 
    plugins=[DestinationsPlugin()],
    name="TravelAgent",
    instructions="You are a helpful AI Agent that can help plan vacations for customers at random destinations",
)

## ഏജന്റുകളെ നടത്തുന്നത്

ഇപ്പോൾ നാം `ChatHistory` ക്രമീകരിച്ചിട്ട് അതിനുള്ളിൽ `system_message` ഉൾപ്പെടുത്തിക്കൊണ്ട് ഏജന്റിനെ 실행ിക്കാം. മുമ്പ് നമുക്ക് സ്ഥാപിച്ച `AGENT_INSTRUCTIONS` നാം ഉപയോഗിക്കും.

ഇവ ക്രമീകരിച്ച ശേഷം, നാം `user_inputs` സൃഷ്ടിക്കുന്നു, ഇത് ഉപയോഗकर्ता ഏജന്റിന് അയക്കുന്ന സന്ദേശത്തെ പ്രതിനിധീകരിക്കുന്നു. ഈ ഉദാഹരണത്തിൽ, സന്ദേശം `Plan me a sunny vacation` എന്നിങ്ങനെ സജ്ജീകരിച്ചിരിക്കുന്നു.

ഈ സന്ദേശം നിങ്ങൾ മാറ്റി, ഏജന്റ് വ്യത്യസ്തമായി പ്രതികരിക്കുന്നത് കാണാം.


In [ ]:
user_inputs = [
    "Plan me a day trip.",
    "I don't like that destination. Plan me another vacation.",
]

async def main():
    thread: ChatHistoryAgentThread | None = None

    for user_input in user_inputs:
        html_output = (
            f"<div style='margin-bottom:10px'>"
            f"<div style='font-weight:bold'>User:</div>"
            f"<div style='margin-left:20px'>{user_input}</div></div>"
        )

        agent_name = None
        full_response: list[str] = []
        function_calls: list[str] = []

        # Buffer to reconstruct streaming function call
        current_function_name = None
        argument_buffer = ""

        async for response in agent.invoke_stream(
            messages=user_input,
            thread=thread,
        ):
            thread = response.thread
            agent_name = response.name
            content_items = list(response.items)

            for item in content_items:
                if isinstance(item, FunctionCallContent):
                    if item.function_name:
                        current_function_name = item.function_name

                    # Accumulate arguments (streamed in chunks)
                    if isinstance(item.arguments, str):
                        argument_buffer += item.arguments
                elif isinstance(item, FunctionResultContent):
                    # Finalize any pending function call before showing result
                    if current_function_name:
                        formatted_args = argument_buffer.strip()
                        try:
                            parsed_args = json.loads(formatted_args)
                            formatted_args = json.dumps(parsed_args)
                        except Exception:
                            pass  # leave as raw string

                        function_calls.append(f"Calling function: {current_function_name}({formatted_args})")
                        current_function_name = None
                        argument_buffer = ""

                    function_calls.append(f"\nFunction Result:\n\n{item.result}")
                elif isinstance(item, StreamingTextContent) and item.text:
                    full_response.append(item.text)

        if function_calls:
            html_output += (
                "<div style='margin-bottom:10px'>"
                "<details>"
                "<summary style='cursor:pointer; font-weight:bold; color:#0066cc;'>Function Calls (click to expand)</summary>"
                "<div style='margin:10px; padding:10px; background-color:#f8f8f8; "
                "border:1px solid #ddd; border-radius:4px; white-space:pre-wrap; font-size:14px; color:#333;'>"
                f"{chr(10).join(function_calls)}"
                "</div></details></div>"
            )

        html_output += (
            "<div style='margin-bottom:20px'>"
            f"<div style='font-weight:bold'>{agent_name or 'Assistant'}:</div>"
            f"<div style='margin-left:20px; white-space:pre-wrap'>{''.join(full_response)}</div></div><hr>"
        )

        display(HTML(html_output))

await main()

---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**പ്രതിജ്ഞാപത്രം**:  
ഈ പ്രമാണം AI പരിഭാഷ സേവനം [Co-op Translator](https://github.com/Azure/co-op-translator) ഉപയോഗിച്ച് പരിഭാഷ ചെയ്തിരിക്കുന്നു. നീറനിർവഹണത്തിനായി ഞങ്ങൾ ശ്രമിച്ചെങ്കിലും, ഓട്ടോമാറ്റിക് പരിഭാഷകളിൽ പിശകുകളും അസംബന്ധതകളും ഉണ്ടാകാമെന്ന് ദയവായി ശ്രദ്ധിക്കുക. മാതൃഭാഷയിലുള്ള اصل പ്രമാണം പ്രാമാണിക സ്രോതസ്സായി കണക്കാക്കപ്പെടണം. അത്യാവശ്യ വിവരങ്ങൾക്ക് പ്രൊഫഷണൽ മാനവപരിഭാഷ നിർദേശിക്കപ്പെടുന്നു. ഈ പരിഭാഷയുടെ ഉപയോഗത്തെ അടിസ്ഥാനമാക്കി ഉണ്ടാകുന്ന തെറ്റിദ്ധാരണകൾക്കോ വിവർത്തനദോഷങ്ങൾക്ക് ഞങ്ങൾ ഉത്തരവാദികളല്ല.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
